# Lesson3 : Dense Retrival

In [76]:
import warnings

# Disable all warnings
warnings.filterwarnings("ignore")

import os
import weaviate
import cohere
from dotenv import load_dotenv
from utils import print_result
from utils import keyword_search

load_dotenv()

# This is a public Server
os.environ['WEAVIATE_API_URL'] = 'https://yvi3vi7ytcqjshbpzpeta.c0.us-west3.gcp.weaviate.cloud/'
os.environ['WEAVIATE_API_KEY'] = 'Qlh3QzlUa0M3ZEtKelprcV9VSTBlRHcwRU1YZUVrOU5TZ3dxMENGWXMvWHMreDBDRHlJYWpjd3lpQWEwPV92MjAw'
co = cohere.Client(os.environ['COHERE_API_KEY'])
auth_config = weaviate.auth.AuthApiKey(
    api_key=os.environ['WEAVIATE_API_KEY'])

In [77]:
client = weaviate.Client(
    url=os.environ['WEAVIATE_API_URL'],
    auth_client_secret=auth_config,
    additional_headers={
        "X-Cohere-Api-Key": os.environ['COHERE_API_KEY'],
    }
)
print(client.is_ready())

True


# Part 1: Vector Database for semantic Search

In [78]:
def dense_retrieval(
    query: str,
    results_lang: str | None = None,
    properties=("text", "title", "url", "lang"),
    num_results: int = 5,
):
    # optional language filter (disable until you know what langs exist)
    where_filter = None
    if results_lang:
        where_filter = {"path": ["lang"], "operator": "Equal", "valueString": results_lang}

    # Try dense with your own query vector (Cohere v3 requires input_type)
    hits = []
    try:
        q_vec = co.embed(
            model="embed-multilingual-v3.0",
            input_type="search_query",
            texts=[query]
        ).embeddings[0]

        q = client.query.get("Articles", list(properties)).with_limit(num_results)
        if where_filter: q = q.with_where(where_filter)
        resp = q.with_near_vector({"vector": q_vec}).do()
        if "errors" in resp: raise RuntimeError(resp["errors"])
        hits = resp.get("data", {}).get("Get", {}).get("Articles", []) or []
    except Exception:
        hits = []

    # Fallback 1: BM25 with same filter
    if not hits:
        q = client.query.get("Articles", list(properties)).with_limit(num_results)
        if where_filter: q = q.with_where(where_filter)
        resp_kw = q.with_bm25(query=query, properties=["title", "text"]).do()
        if "errors" in resp_kw: raise RuntimeError(resp_kw["errors"])
        hits = resp_kw.get("data", {}).get("Get", {}).get("Articles", []) or []

    # Fallback 2: BM25 without filter (guarantee *something*)
    if not hits and where_filter:
        resp_kw2 = (
            client.query.get("Articles", list(properties))
            .with_bm25(query=query, properties=["title", "text"])
            .with_limit(num_results)
            .do()
        )
        if "errors" in resp_kw2: raise RuntimeError(resp_kw2["errors"])
        hits = resp_kw2.get("data", {}).get("Get", {}).get("Articles", []) or []

    return hits

In [79]:
for q in [
    "Who wrote Hamlet?",
    "What is the capital of Canada?",
    "Tallest person in history?",
    "film about a time travel paradox",
    "أطول رجل في التاريخ"
]:
    print("Q:", q)
    res = dense_retrieval(q)   # or results_lang='en' if you confirmed langs
    print_result(res)

Q: Who wrote Hamlet?
Q: What is the capital of Canada?
Q: Tallest person in history?
Q: film about a time travel paradox
Q: أطول رجل في التاريخ
item 0
lang:ar

text:نهائي كأس العالم FIFA يعد من أكثر الأحداث التلفزيونية مشاهدةً في العالم.

title:نهائي كأس العالم

url:https://example.com/fifa-final-ar


item 1
lang:ar

text:نهائي كأس العالم FIFA يعد من أكثر الأحداث التلفزيونية مشاهدةً في العالم.

title:نهائي كأس العالم

url:https://example.com/fifa-final-ar




# Document embeddings
response = co.embed(
    model="embed-multilingual-v3.0",
    input_type="search_document",
    texts=texts.tolist()
).embeddings
embeds = np.array(response)

def search(query):
    query_embed = co.embed(
        model="embed-multilingual-v3.0",
        input_type="search_query",
        texts=[query]
    ).embeddings[0]
    ids, dists = search_index.get_nns_by_vector(query_embed, 3, include_distances=True)
    return pd.DataFrame({"texts": texts[ids], "distance": dists})

In [80]:
probe = (
    client.query
    .get("Articles", ["title"])
    .with_additional(["id", "vector"])
    .with_limit(1)
    .do()
)

print(json.dumps(probe, indent=2))

query = "Who wrote Hamlet?"
dense_retrieval_results = dense_retrieval(query)
print_result(dense_retrieval_results)

{
  "data": {
    "Get": {
      "Articles": [
        {
          "_additional": {
            "id": "04914ded-8f23-4589-92e6-d83dbcd44fa4",
            "vector": []
          },
          "title": "Final de la Copa Mundial"
        }
      ]
    }
  }
}


In [81]:
query = "What is the capital of Canada?"
dense_retrieval_results = dense_retrieval(query)
print_result(dense_retrieval_results)

In [82]:
query = "Tallest person in history?"
keyword_search_results = keyword_search(query, client)
print_result(keyword_search_results)

In [83]:
query = "Tallest person in history"
dense_retrieval_results = dense_retrieval(query)
print_result(dense_retrieval_results)

In [84]:
query = "أطول رجل في التاريخ"
dense_retrieval_results = dense_retrieval(query)
print_result(dense_retrieval_results)

item 0
lang:ar

text:نهائي كأس العالم FIFA يعد من أكثر الأحداث التلفزيونية مشاهدةً في العالم.

title:نهائي كأس العالم

url:https://example.com/fifa-final-ar


item 1
lang:ar

text:نهائي كأس العالم FIFA يعد من أكثر الأحداث التلفزيونية مشاهدةً في العالم.

title:نهائي كأس العالم

url:https://example.com/fifa-final-ar




In [85]:
query = "film about a time travel paradox"
dense_retrieval_results = dense_retrieval(query)
print_result(dense_retrieval_results)

# Part 2: Building Semantic Search from Scratch

In [86]:
from annoy import AnnoyIndex
import numpy as np
import pandas as pd

In [87]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

In [88]:
texts = text.split('.')
texts = np.array([t.strip(' \n') for t in texts])
texts

array(['Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan',
       'It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine',
       'Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind',
       'Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007',
       'Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar',
       'Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm',
       'Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles',

In [89]:
texts = text.split('\n\n')
texts = np.array([t.strip(' \n') for t in texts])
texts

array(['Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.\nIt stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.\nSet in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.',
       'Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.\nCaltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.\nCinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.\nPrincipal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.\nInterstellar uses extensive practical 

In [90]:
texts = text.split('.')
texts = np.array([t.strip(' \n') for t in texts])

title = 'Interstellar (film)'

texts = np.array([f"{title} {t}" for t in texts])

texts

array(['Interstellar (film) Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan',
       'Interstellar (film) It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine',
       'Interstellar (film) Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind',
       'Interstellar (film) Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007',
       'Interstellar (film) Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar',
       'Interstellar (film) Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format

In [91]:
# Get the embeddings:

In [92]:
response = co.embed(
    texts=texts.tolist()
).embeddings

In [93]:
embeds = np.array(response)
embeds.shape

(15, 4096)

In [94]:
# Create the search index:

In [95]:
search_index = AnnoyIndex(embeds.shape[1], 'angular')
# Add all the vectors to the search index
for i in range(len(embeds)):
    search_index.add_item(i, embeds[i])

search_index.build(10) # 10 trees
search_index.save('test.ann')

True

In [96]:
pd.set_option('display.max_colwidth', None)

def search(query):

  # Get the query's embedding
  query_embed = co.embed(texts=[query]).embeddings

  # Retrieve the nearest neighbors
  similar_item_ids = search_index.get_nns_by_vector(query_embed[0],
                                                    3,
                                                  include_distances=True)
  # Format the results
  results = pd.DataFrame(data={'texts': texts[similar_item_ids[0]],
                              'distance': similar_item_ids[1]})

  print(texts[similar_item_ids[0]])
    
  return results

In [97]:
query = "How much did the film make?"
search(query)

['Interstellar (film) The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'
 'Interstellar (film) Interstellar premiered on October 26, 2014, in Los Angeles'
 'Interstellar (film) In the United States, it was first released on film stock, expanding to venues using digital projectors']


,texts,distance
0,"Interstellar (film) The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014",1.019055
1,"Interstellar (film) Interstellar premiered on October 26, 2014, in Los Angeles",1.144950
2,"Interstellar (film) In the United States, it was first released on film stock, expanding to venues using digital projectors",1.167268
